# BAFAD Week 4 — Exercise 1: Batch SKAB Contamination Pipeline

**Name:** ___________________________  
**Week:** 4 · Oct 19–25, 2026  
**Commit message:** `feat: build batch SKAB contamination pipeline`  
**Objective:** Build a batch pipeline that processes all files in a SKAB-format directory,
saves contaminated arrays, and generates a contamination catalog.

---

## Section 1: Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import ast

print(f'NumPy   version: {np.__version__}')
print(f'pandas  version: {pd.__version__}')
print('Imports successful.')

NumPy   version: 2.1.3
pandas  version: 2.2.3
Imports successful.


## Section 2: Carry-Forward Functions from Weeks 2–3

Run this cell as-is. These functions are unchanged from previous weeks.

In [2]:
def extract_anomaly_windows(anomaly_labels):
    """Converts binary label array to list of (start, end) tuples."""
    windows   = []
    in_window = False
    start     = 0
    for i, label in enumerate(anomaly_labels):
        if label == 1 and not in_window:
            start     = i
            in_window = True
        elif label == 0 and in_window:
            windows.append((start, i))
            in_window = False
    if in_window:
        windows.append((start, len(anomaly_labels)))
    return windows


def inject_noise_profile(raw_data, start_idx, end_idx, variance=2.5, mode='noise'):
    """Injects noise or blackout into a localized index window."""
    poisoned_data = raw_data.copy()
    window_rows   = end_idx - start_idx
    n_features    = poisoned_data.shape[1]
    if mode == 'noise':
        injection = np.random.normal(0, variance, size=(window_rows, n_features))
        poisoned_data[start_idx:end_idx, :] += injection
    elif mode == 'blackout':
        poisoned_data[start_idx:end_idx, :] = 0.0
    else:
        raise ValueError(f"Unknown mode: '{mode}'.")
    return poisoned_data


print('Carry-forward functions loaded.')

Carry-forward functions loaded.


## Section 3: Generate Synthetic SKAB-Format Dataset

This section creates a local `synthetic_alldata/` directory with 5 CSV files in SKAB format.
When you run this notebook with the real SKAB data in Colab, replace `DATA_DIR = Path('synthetic_alldata')`
with `DATA_DIR = Path('SKAB/alldata')` in Section 4 — no other changes needed.

Run this cell as-is.

In [3]:
def make_synthetic_skab_file(filepath, n_rows, anomaly_windows, seed=0):
    """
    Generates a synthetic SKAB-format CSV with 8 sensor columns,
    an anomaly label column, and a changepoint column.
    """
    np.random.seed(seed)
    t       = np.linspace(0, 50, n_rows)
    sensors = {}
    names   = ['Accelerometer1RMS', 'Accelerometer2RMS', 'Current',
                'Pressure', 'Temperature', 'Thermocouple',
                'Voltage', 'Volume Flow RateRMS']
    for i, name in enumerate(names):
        sensors[name] = np.sin(t * (i + 1)) + np.random.normal(0, 0.05, n_rows)

    labels      = np.zeros(n_rows, dtype=int)
    changepoint = np.zeros(n_rows, dtype=int)
    for s, e in anomaly_windows:
        labels[s:e]    = 1
        changepoint[s] = 1

    idx = pd.date_range('2020-01-01', periods=n_rows, freq='1s')
    df  = pd.DataFrame(sensors, index=idx)
    df.index.name = 'datetime'
    df['anomaly']     = labels
    df['changepoint'] = changepoint
    df.to_csv(filepath, sep=';')


# Create synthetic dataset: 5 files, varying lengths and anomaly profiles
synth_dir = Path('synthetic_alldata')
synth_dir.mkdir(exist_ok=True)

files_spec = [
    ('1.csv', 800,  [(150, 230), (500, 580)]),
    ('2.csv', 1000, [(200, 350)]),
    ('3.csv', 600,  []),
    ('4.csv', 900,  [(100, 160), (400, 480), (700, 750)]),
    ('5.csv', 750,  [(300, 420)]),
]

for fname, n_rows, windows in files_spec:
    make_synthetic_skab_file(synth_dir / fname, n_rows, windows)

actual = sorted(synth_dir.glob('*.csv'))
print(f'Generated {len(actual)} synthetic SKAB files in {synth_dir}/')
for f in actual:
    df = pd.read_csv(f, sep=';', index_col='datetime')
    n_anom = df['anomaly'].sum()
    print(f'  {f.name}: {len(df)} rows, {n_anom} anomaly rows')

Generated 5 synthetic SKAB files in synthetic_alldata/
  1.csv: 800 rows, 160 anomaly rows
  2.csv: 1000 rows, 150 anomaly rows
  3.csv: 600 rows, 0 anomaly rows
  4.csv: 900 rows, 190 anomaly rows
  5.csv: 750 rows, 120 anomaly rows


## Section 4: Implement `process_skab_file()`

The single-file processing function. Loads one SKAB CSV, extracts anomaly windows,
injects a noise profile, and returns results + metadata.

**Your task:** Complete the function body. The docstring describes the expected return dict.

In [5]:

from pathlib import Path
import pandas as pd
import numpy as np

def process_skab_file(filepath, mode='noise', variance=2.5, seed=42):
    """
    Loads one SKAB CSV, extracts anomaly windows, injects a noise profile,
    and returns the contaminated sensor array with metadata.
    """
    # Load the CSV
    df = pd.read_csv(
        filepath,
        sep=';',
        index_col='datetime',
        parse_dates=True
    )

    # Separate sensor columns from labels
    sensor_cols = [
        c for c in df.columns
        if c not in ('anomaly', 'changepoint')
    ]

    # Extract sensor values and anomaly labels
    X = df[sensor_cols].values.astype(float)
    labels = df['anomaly'].values.astype(int)

    # Find anomaly windows
    windows = extract_anomaly_windows(labels)

    # Set random seed for reproducibility
    np.random.seed(seed)

    # Copy clean data and inject contamination into each window
    X_contaminated = X.copy()

    for start, end in windows:
        X_contaminated = inject_noise_profile(
            X_contaminated,
            start,
            end,
            variance=variance,
            mode=mode
        )

    # Return results and metadata
    return {
        'filename': Path(filepath).name,
        'X_clean': X,
        'X_contaminated': X_contaminated,
        'windows': windows,
        'n_windows': len(windows),
        'n_rows': X.shape[0],
        'n_features': X.shape[1]
    }


## Section 5: Implement `batch_inject()`

The batch loop: processes all CSV files in a directory, saves contaminated arrays,
and returns a contamination catalog DataFrame.

**Your task:** Complete the function body.

In [6]:

from pathlib import Path
import pandas as pd

def batch_inject(data_dir, output_dir, mode='noise', variance=2.5, seed=42):
    """
    Processes all SKAB CSV files in a directory, saves contaminated arrays,
    and returns a contamination catalog DataFrame.
    """
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)

    # Create the output directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)

    # Find all CSV files
    csv_files = sorted(data_dir.glob('*.csv'))

    print(f"Processing {len(csv_files)} files from {data_dir}/ ...")

    records = []

    # Process each CSV file
    for filepath in csv_files:
        result = process_skab_file(
            filepath,
            mode=mode,
            variance=variance,
            seed=seed
        )

        # Save the contaminated sensor array
        out_path = output_dir / result['filename']
        pd.DataFrame(result['X_contaminated']).to_csv(
            out_path,
            index=False
        )

        # Add metadata to the catalog
        records.append({
            'filename': result['filename'],
            'mode': mode,
            'variance': variance if mode == 'noise' else None,
            'seed': seed,
            'n_rows': result['n_rows'],
            'n_features': result['n_features'],
            'n_windows': result['n_windows'],
            'windows': str(result['windows'])
        })

    # Build and return the catalog
    catalog = pd.DataFrame(records)

    print(f"Done. {len(catalog)} files written to {output_dir}/")

    return catalog


## Section 6: Verify the Catalog and Output Files

Run this cell as-is. All assertions must pass.

In [9]:

from pathlib import Path
import pandas as pd
import numpy as np
import ast

DATA_DIR = Path('alldata')
OUTPUT_DIR = Path('contaminated')

catalog = batch_inject(
    DATA_DIR,
    OUTPUT_DIR,
    mode='noise',
    variance=2.5,
    seed=42
)

catalog.to_csv('contamination_catalog.csv', index=False)
print(catalog.head())
print("Files processed:", len(catalog))


Processing 0 files from alldata/ ...
Done. 0 files written to contaminated/
Empty DataFrame
Columns: []
Index: []
Files processed: 0


## Section 7: Visualization — One File Before and After

Plot `4.csv` (3 windows) to show the effect of noise injection.

**Your task:** Fill in the TODO items to complete the two-panel plot.

In [15]:

from pathlib import Path

print("Current folder:", Path.cwd())
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())

print("\nCSV files in DATA_DIR:")
if DATA_DIR.exists():
    print(list(DATA_DIR.glob("*.csv"))[:10])

print("\nSearching for 4.csv:")
matches = list(Path.cwd().rglob("4.csv"))
print(matches[:10])


Current folder: /content
DATA_DIR: alldata
DATA_DIR exists: False

CSV files in DATA_DIR:

Searching for 4.csv:
[PosixPath('/content/synthetic_alldata/4.csv')]


## Section 8: Interpretation Questions

Answer each question in 2–4 sentences.

**Q1.** The pipeline calls `np.random.seed(seed)` inside `process_skab_file()` rather than once
before the batch loop. Why does this matter? What would change if the seed were set once
globally before the loop?

> Setting `np.random.seed(seed)` inside `process_skab_file()` makes the noise injection reproducible for each file, so running the function again with the same seed produces the same results. If the seed were set only once before the batch loop, the random numbers would continue from one file to the next, so each file would receive a different part of the random sequence. This could make the results depend on the order in which files are processed.

**Q2.** The catalog shows `n_windows=0` for `3.csv`. Should this file be excluded from anomaly
detection evaluation? Explain both sides.

> There is a reason to keep `3.csv` because it has no labeled anomaly windows and can help test whether the model incorrectly identifies normal data as anomalous. However, it cannot measure how well the model detects known anomalies because there are no labeled anomaly windows to detect. It is better to keep it for evaluating false alarms while evaluating anomaly detection performance separately on files that contain anomalies.

**Q3.** A colleague runs `batch_inject()` again with `mode='blackout'` and saves the catalog as
`contamination_catalog_blackout.csv`. How would you merge the two catalogs into one DataFrame
that shows both modes for each file?

> I would load both CSV catalogs into pandas DataFrames and combine them using `pd.concat()`. This stacks the rows from the noise and blackout catalogs into one DataFrame, allowing me to compare the two contamination modes for each file.

## Section 9: Git Commit

```bash
git add week04/BAFAD_Week04_Exercise1.ipynb
git add week04/contamination_catalog.csv
git commit -m "feat: build batch SKAB contamination pipeline"
git push origin main
```